# Multi-Paradigm Bug Fixing

This notebook includes:
- SentencePiece Tokenizer Training
- T5-small Pretraining
- 4 Unique Paradigms for AI Bug Fixing:
    - fine-tuning with pretraining
    - fine-tuning without pretraining
    - RAG
    - zero-shot
- Evaluation and Cross-Analysis of all Paradigms


## 1. Environment Setup
- Install dependencies
- Mount Google Drive if needed
- Set paths
- Import project modules
- Set seed and device


In [3]:
import sys
from pathlib import Path

REPO_URL = "https://github.com/sambennett04/multi-paradigm-bug-fixing.git"
REPO_DIR = Path("/content/multi-paradigm-bug-fixing")

# Clone repo if it doesn't exist, otherwise pull latest
if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

# Add repo root to Python path (so `from src...` works)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

# Install dependencies
%pip install -r {REPO_DIR}/requirements.txt

# (Optional) Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Debug prints (sanity check)
print("Repo dir:", REPO_DIR)
print("src exists:", (REPO_DIR / "src").exists())
print("data_utils exists:", (REPO_DIR / "src" / "data_utils.py").exists())

Already up to date.
Mounted at /content/drive
Repo dir: /content/multi-paradigm-bug-fixing
src exists: True
data_utils exists: True


In [4]:
from pathlib import Path
import sys

# Repo root (always correct after %cd)
PROJECT_ROOT = Path.cwd()
print("Project root:", PROJECT_ROOT)

#Add repo to sys path so that imported functions resolve correctly
sys.path.append(str(PROJECT_ROOT))

# Persistent storage (Google Drive)
DRIVE_ROOT = Path("/content/drive/MyDrive/multi-paradigm-bug-fixing")
print("Drive root:", DRIVE_ROOT)

DATA_DIR = DRIVE_ROOT / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Initialize intermediary and final output folders in mounted google drive
OUTPUTS_DIR = DRIVE_ROOT / "outputs"

TOKENIZER_DIR = OUTPUTS_DIR / "tokenizer"
PRETRAIN_DIR = OUTPUTS_DIR / "pretrain_model"
FINETUNE_A_DIR = OUTPUTS_DIR / "finetune_a"
FINETUNE_B_DIR = OUTPUTS_DIR / "finetune_b"
KNOWLEDGE_BASE_DIR = OUTPUTS_DIR / "knowledge_base"
PREDICTIONS_DIR = OUTPUTS_DIR / "predictions"
RESULTS_DIR = OUTPUTS_DIR / "results"

# Create directories
for path in [
    TOKENIZER_DIR,
    PRETRAIN_DIR,
    FINETUNE_A_DIR,
    FINETUNE_B_DIR,
    KNOWLEDGE_BASE_DIR,
    PREDICTIONS_DIR,
    RESULTS_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)

print("Outputs will be saved to:", OUTPUTS_DIR)


Project root: /content
Drive root: /content/drive/MyDrive/multi-paradigm-bug-fixing
Outputs will be saved to: /content/drive/MyDrive/multi-paradigm-bug-fixing/outputs


In [5]:
import random
import torch

#setting random seed for math.random, torch and torch cuda
random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

### Helper Method to Skip Phase if Output Already Populated


In [6]:
def output_exists(output_file_path):
  if output_file_path.exists():
    print(f"Output file {output_file_path} already exists.")
    return True
  else:
    return False

## 2. Load and Prepare Pretraining Data
- Load CodeSearchNet Java
- Sample ~50K methods
- Extract method bodies
- Preview samples


In [7]:
import json
from src.data_utils import fetch_pretraining_data_hugging_face, save_pretraining_data, load_pretraining_data

pretraining_save_path = DATA_DIR / "pretraining_methods.txt"

if not output_exists(pretraining_save_path):
  #load only the train split of CodeSearchNet, take random sample of 50,000 methods and extract their bodies
  #Print a summary record of the dataset loading with 3 example methods
  pretraining_methods = fetch_pretraining_data_hugging_face(n_samples=50000, random_seed=42, audit_sample_count=3)

  #cache extracted method bodies in a txt file with one flattened function per line
  #this format is in line with what SentencePieceTrainer expects in the next section
  save_pretraining_data(pretraining_save_path=pretraining_save_path, pretraining_methods= pretraining_methods)

#load flattened pretraining data from save file to guarantee constistency across first and repeat runs
pretraining_methods = load_pretraining_data(pretraining_save_path)

Output file /content/drive/MyDrive/multi-paradigm-bug-fixing/data/pretraining_methods.txt already exists.


## 3. Train/Load SentencePiece Tokenizer and Load Roberta Tokenizer
- Train SentencePiece tokenizer once
  - MUST include T5-required special tokens
- Save tokenizer
- reload and convert tokenizer to hugging face


In [8]:
from src.tokenizer_utils import train_tokenizer, load_tokenizer
from transformers import RobertaTokenizer

#train tokenizer and save model/vocab files
corpus_file = DATA_DIR / "pretraining_methods.txt"
model_prefix = TOKENIZER_DIR / "tokenizer"

model_file_exists = output_exists(model_prefix.with_suffix(".model"))
vocab_file_exists = output_exists(model_prefix.with_suffix(".vocab"))

if not model_file_exists and not vocab_file_exists:
  train_tokenizer(corpus_file, model_prefix)

#load trained sentence piece tokenizer as HuggingFace unigram model
sentencepiece_tokenizer = load_tokenizer(model_prefix)

Output file /content/drive/MyDrive/multi-paradigm-bug-fixing/outputs/tokenizer/tokenizer.model already exists.
Output file /content/drive/MyDrive/multi-paradigm-bug-fixing/outputs/tokenizer/tokenizer.vocab already exists.


## 4. Build 2 T5-small Model's, One For Each Experiment
- Define T5Config manually
- Initialize model from scratch
- Resize token embeddings
- Verify parameter count


In [9]:
from src.model_utils import build_t5_model

#build two "fres" t5 models from scratch with same parameters
fresh_model_A = build_t5_model(tokenizer=sentencepiece_tokenizer, embd_hidd_dim = 512, feed_forward_dim = 2048, key_val_proj_dim = 64, num_heads = 8, num_encoder_layers = 6, num_decoder_layers = 6)
fresh_model_B = build_t5_model(tokenizer=sentencepiece_tokenizer, embd_hidd_dim = 512, feed_forward_dim = 2048, key_val_proj_dim = 64, num_heads = 8, num_encoder_layers = 6, num_decoder_layers = 6)

print(f"Fresh Model Shape: {fresh_model_A.shared.weight.shape}")

Fresh Model Shape: torch.Size([16384, 512])


## 5. Pretraining (For Pipeline A)
- Apply span corruption
- Build pretraining dataset
- Train for 3 epochs
- Save final pretrained model

In [10]:
from src.pretrain_utils import (
    PretrainConfig,
    build_pretrain_dataset,
    run_pretraining,
    filter_by_length
)
from transformers import get_linear_schedule_with_warmup, T5ForConditionalGeneration
from math import ceil

config_exists = output_exists(PRETRAIN_DIR / "config.json")
gen_config_exists = output_exists(PRETRAIN_DIR / "generation_config.json")
model_exists = output_exists(PRETRAIN_DIR / "model.safetensors")

if not (config_exists and gen_config_exists and model_exists):
  #define configuraton for pretraining
  pretrain_config = PretrainConfig(
      output_dir=PRETRAIN_DIR,
      min_tokens = 10,
      max_tokens = 512,
      max_samples = None, #you may set sample limits for smoke testing with this value
      corruption_rate = 0.15,
      num_epochs = 3,
      batch_size = 8,
      device = "cuda" if torch.cuda.is_available() else "cpu",
  )

  #instantiate optimizer
  optimizer = torch.optim.AdamW(
      fresh_model_A.parameters(),
      lr=5e-4,
      weight_decay=0.01,
  )

  #Calculate num training steps so that the scheduler can properly decay learning rate of training
  effective_pretraining_methods = pretraining_methods
  if pretrain_config.max_samples is not None:
      effective_pretraining_methods = effective_pretraining_methods[:pretrain_config.max_samples]

  length_tokenized_filtered_methods = 0
  for method in effective_pretraining_methods:
      token_ids = sentencepiece_tokenizer.encode(method, add_special_tokens=False)
      if filter_by_length(token_ids, pretrain_config=pretrain_config):
          length_tokenized_filtered_methods += 1

  steps_per_epoch = ceil(length_tokenized_filtered_methods / pretrain_config.batch_size)
  num_training_steps = pretrain_config.num_epochs * steps_per_epoch

  #instantiate scheduler with num training steps for proper lr decay
  scheduler = get_linear_schedule_with_warmup(
      optimizer=optimizer,
      num_warmup_steps=0,
      num_training_steps=num_training_steps,
  )

  #move model to divice and run pretraining
  fresh_model_A.to(pretrain_config.device)

  run_pretraining(raw_pretraining_methods = pretraining_methods, model = fresh_model_A, optimizer = optimizer, tokenizer = sentencepiece_tokenizer, scheduler = scheduler, pretrain_config = pretrain_config)
else:
  print(f"Pretrained model files found. Loading model from {PRETRAIN_DIR}...")
  fresh_model_A = T5ForConditionalGeneration.from_pretrained(str(PRETRAIN_DIR))
  fresh_model_A.to("cuda" if torch.cuda.is_available() else "cpu")


Output file /content/drive/MyDrive/multi-paradigm-bug-fixing/outputs/pretrain_model/config.json already exists.
Output file /content/drive/MyDrive/multi-paradigm-bug-fixing/outputs/pretrain_model/generation_config.json already exists.
Output file /content/drive/MyDrive/multi-paradigm-bug-fixing/outputs/pretrain_model/model.safetensors already exists.
Pretrained model files found. Loading model from /content/drive/MyDrive/multi-paradigm-bug-fixing/outputs/pretrain_model...


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

## 6. Load Fine-tuning Data
- Load CodeXGLUE bug-fix dataset
- Prepare train / validation / test splits
- Inspect example pair


In [11]:
from src.data_utils import fetch_finetuning_data_hugging_face, save_finetuning_data, load_finetuning_data

finetuning_train_save_path = DATA_DIR / "finetuning_train_method_pairs.txt"
finetuning_valid_save_path = DATA_DIR / "finetuning_valid_method_pairs.txt"

finetuning_train_data_existis = output_exists(finetuning_train_save_path)
finetuning_valid_data_existis = output_exists(finetuning_valid_save_path)

if not (finetuning_train_data_existis and finetuning_valid_data_existis):

  #fetch the buggy and fixed method pairs from codex glue code refinement train and validation split
  finetuning_train_method_pairs, finetuning_valid_method_pairs = fetch_finetuning_data_hugging_face(audit_sample_count=1)

  #save fixed/buggy pairs as json list of list([[buggy, fixed], [buggy, fixed], ...])
  save_finetuning_data(finetuning_save_path=finetuning_train_save_path, finetuning_method_pairs=finetuning_train_method_pairs)
  save_finetuning_data(finetuning_save_path=finetuning_valid_save_path, finetuning_method_pairs=finetuning_valid_method_pairs)

#load train and validation pairs for finetuning
finetuning_train_method_pairs = load_finetuning_data(finetuning_train_save_path)
finetuning_valid_method_pairs = load_finetuning_data(finetuning_valid_save_path)

Output file /content/drive/MyDrive/multi-paradigm-bug-fixing/data/finetuning_train_method_pairs.txt already exists.
Output file /content/drive/MyDrive/multi-paradigm-bug-fixing/data/finetuning_valid_method_pairs.txt already exists.


## 7i. Finetune Pre-trained t5 Model
- Load final pretrained checkpoint
- build fine-tuning dataset
- Train on bug-fixing task
- Generate and save test predictions


In [12]:
from math import ceil
from transformers import T5ForConditionalGeneration, get_linear_schedule_with_warmup
from src.finetune_utils import run_finetuning, FinetuneConfig

if not output_exists(FINETUNE_A_DIR / "model.safetensors"):
    #fine tuning config
    finetune_config = FinetuneConfig(
        output_dir=FINETUNE_A_DIR,
        max_input_len = 512,
        max_target_len = 512,

        max_train_samples = None, #set max train samples
        max_val_samples = None, #set max val samples
        num_epochs = 3,
        batch_size = 8,
        device = "cuda" if torch.cuda.is_available() else "cpu"
    )

    # load model pretrained in step 5
    pretrained_t5_model = T5ForConditionalGeneration.from_pretrained(str(PRETRAIN_DIR))

    #instantiate optimizer
    optimizer = torch.optim.AdamW(
        pretrained_t5_model.parameters(),
        lr=5e-4,
        weight_decay=0.01,
    )

    #scheduler uses num_training_steps to decide how to change the learning rate over time (linearly decay learning rate toward zero over all training steps)
    effective_train_pairs = finetuning_train_method_pairs
    if finetune_config.max_train_samples is not None:
        effective_train_pairs = effective_train_pairs[:finetune_config.max_train_samples]

    num_train_examples = len(effective_train_pairs)
    steps_per_epoch = ceil(num_train_examples / finetune_config.batch_size)
    num_training_steps = finetune_config.num_epochs * steps_per_epoch

    #instantiate scheduler with num steps
    scheduler = get_linear_schedule_with_warmup(
        optimizer=optimizer,
        num_warmup_steps=0,
        num_training_steps=num_training_steps,
    )

    #move model to device and finetune
    pretrained_t5_model.to(finetune_config.device)

    run_finetuning(
        train_pairs=finetuning_train_method_pairs,
        val_pairs=finetuning_valid_method_pairs,
        model=pretrained_t5_model,
        optimizer=optimizer,
        tokenizer=sentencepiece_tokenizer,
        scheduler=scheduler,
        finetune_config=finetune_config,
    )
else:
    print(f"Finetuned model A already exists at {FINETUNE_A_DIR}. Skipping training.")


Output file /content/drive/MyDrive/multi-paradigm-bug-fixing/outputs/finetune_a/model.safetensors already exists.
Finetuned model A already exists at /content/drive/MyDrive/multi-paradigm-bug-fixing/outputs/finetune_a. Skipping training.


## 7ii. Fine-tune Fresh t5 Model
- Use same tokenizer and config
- Train on bug-fixing task
- Generate and save test predictions


In [13]:
from math import ceil
from transformers import T5ForConditionalGeneration, get_linear_schedule_with_warmup
from src.finetune_utils import run_finetuning

if not output_exists(FINETUNE_B_DIR / "model.safetensors"):
    #fine tuning config
    finetune_config = FinetuneConfig(
        output_dir=FINETUNE_B_DIR,
        max_input_len = 512,
        max_target_len = 512,

        max_train_samples = None, #set max train samples
        max_val_samples = None, #set max val samples
        num_epochs = 3,
        batch_size = 8,
        device = "cuda" if torch.cuda.is_available() else "cpu"
    )

    #instantiate optimizer
    optimizer = torch.optim.AdamW(
        fresh_model_B.parameters(),
        lr=5e-4,
        weight_decay=0.01,
    )

    #calculate num_training_steps to be used by the schedulers for both pipelines
    #scheduler uses num_training_steps to decide how to change the learning rate over time (linearly decay learning rate toward zero over all training steps)
    effective_train_pairs = finetuning_train_method_pairs
    if finetune_config.max_train_samples is not None:
        effective_train_pairs = effective_train_pairs[:finetune_config.max_train_samples]

    num_train_examples = len(effective_train_pairs)
    steps_per_epoch = ceil(num_train_examples / finetune_config.batch_size)
    num_training_steps = finetune_config.num_epochs * steps_per_epoch

    #instantiate scheduler with num steps
    scheduler = get_linear_schedule_with_warmup(
        optimizer=optimizer,
        num_warmup_steps=0,
        num_training_steps=num_training_steps,
    )

    #move model to device and finetune
    fresh_model_B.to(finetune_config.device)

    run_finetuning(
        train_pairs=finetuning_train_method_pairs,
        val_pairs=finetuning_valid_method_pairs,
        model=fresh_model_B,
        optimizer=optimizer,
        tokenizer=sentencepiece_tokenizer,
        scheduler=scheduler,
        finetune_config=finetune_config,
    )
else:
    print(f"Finetuned model B already exists at {FINETUNE_B_DIR}. Skipping training.")


Output file /content/drive/MyDrive/multi-paradigm-bug-fixing/outputs/finetune_b/model.safetensors already exists.
Finetuned model B already exists at /content/drive/MyDrive/multi-paradigm-bug-fixing/outputs/finetune_b. Skipping training.


## 8 Initialize QWEN and RAG Knowledge BASE

In [14]:
from src.llm_utils import (
    LLMConfig,
    build_FAISS_index,
    load_qwen_generator,
    save_knowledge_base,
    train_embedding_model_and_generate_embeddings,
)

#Train Code2Vec-style embedding model on buggy methods and embed the corpus
embedding_model, embeddings, metadata = train_embedding_model_and_generate_embeddings(finetuning_train_method_pairs)

print(f"Retained {len(metadata)} parseable training pairs.")
print(f"Embeddings shape: {embeddings.shape}")

#Build FAISS index
faiss_index = build_FAISS_index(embeddings)
print(f"FAISS index size: {faiss_index.ntotal}")

#Save knowledge base
save_knowledge_base(
    embedding_model=embedding_model,
    index=faiss_index,
    metadata=metadata,
    output_dir=KNOWLEDGE_BASE_DIR,
)
print(f"Knowledge base saved to: {KNOWLEDGE_BASE_DIR}")

#Load generator assets
qwen_tokenizer, model = load_qwen_generator()

#Initialize config for later zero-shot / RAG generation
llm_config = LLMConfig(
    embedding_model=embedding_model,
    faiss_index=faiss_index,
    metadata=metadata,
    tokenizer=qwen_tokenizer,
    model=model,
    num_samples=1,
    temperature=0.0,
    max_new_tokens=512,
    top_k=3,
)

print("LLMConfig initialized.")


Streaming output truncated to the last 5000 lines.
/content/multi-paradigm-bug-fixing/src/llm_utils.py:82: RuntimeWarning: Skipping record 13638: buggy method could not be parsed as Java
  _warn_on_skipped_record(index, "buggy method could not be parsed as Java")
/content/multi-paradigm-bug-fixing/src/llm_utils.py:82: RuntimeWarning: Skipping record 13657: buggy method could not be parsed as Java
  _warn_on_skipped_record(index, "buggy method could not be parsed as Java")
/content/multi-paradigm-bug-fixing/src/llm_utils.py:82: RuntimeWarning: Skipping record 13668: buggy method could not be parsed as Java
  _warn_on_skipped_record(index, "buggy method could not be parsed as Java")
/content/multi-paradigm-bug-fixing/src/llm_utils.py:82: RuntimeWarning: Skipping record 13675: buggy method could not be parsed as Java
  _warn_on_skipped_record(index, "buggy method could not be parsed as Java")
/content/multi-paradigm-bug-fixing/src/llm_utils.py:82: RuntimeWarning: Skipping record 13682: bu

Retained 48931 parseable training pairs.
Embeddings shape: (48931, 128)
FAISS index size: 48931
Knowledge base saved to: /content/drive/MyDrive/multi-paradigm-bug-fixing/outputs/knowledge_base


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

LLMConfig initialized.


## 10. Evaluation
- Generate on Test Set
- Compute and Compare Select Metrics


### 10i. Generate Results for ___ Test Set using all models and configurations
- load saved t5 models
- fetch codex glue code refinement Test Set
- calculate results for each configuration
- save to dictionary

In [15]:
from tqdm.auto import tqdm
import torch
import json

from src.data_utils import fetch_test_data_hugging_face
from src.llm_utils import generate_code

predictions_path = PREDICTIONS_DIR / "predictions.json"

#fetch and reduce test pairs from Codex Glue
test_pairs = fetch_test_data_hugging_face()[:500]
buggy_methods = [buggy_method for buggy_method, _ in test_pairs]

if not output_exists(predictions_path):
    #move t5 device if not already there
    device = "cuda" if torch.cuda.is_available() else "cpu"
    pretrained_finetuned_t5 = T5ForConditionalGeneration.from_pretrained(str(FINETUNE_A_DIR)).to(device)
    fresh_finetuned_t5 = T5ForConditionalGeneration.from_pretrained(str(FINETUNE_B_DIR)).to(device)

    #set t5 models to eval mode
    pretrained_finetuned_t5.eval()
    fresh_finetuned_t5.eval()

    #initalize predictions dictionary
    predictions = {buggy_method: {} for buggy_method in buggy_methods}

    #instantiate pre-trained device for moving t5 eval batches to devices
    pretrained_device = next(pretrained_finetuned_t5.parameters()).device
    fresh_device = next(fresh_finetuned_t5.parameters()).device

    batch_size = 8

    # T5 pass in batches
    for i in tqdm(range(0, len(buggy_methods), batch_size), desc="T5 generation"):
        #batch buggy methods from test set with 'fix bug: ' prefix
        batch_buggy = buggy_methods[i:(i + batch_size)]
        batch_text = [f"fix bug: {code}" for code in batch_buggy]

        #huggingface infers batches from the input shape
        #returns batch encoding (basically matric of token ids)
        batch_inputs = sentencepiece_tokenizer(
            batch_text,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512,
        )

        #generate results on batch and store them in predictions dict
        with torch.no_grad():
            pretrained_outputs = pretrained_finetuned_t5.generate(
                input_ids=batch_inputs["input_ids"].to(pretrained_device),
                attention_mask=batch_inputs["attention_mask"].to(pretrained_device),
            )
            fresh_outputs = fresh_finetuned_t5.generate(
                input_ids=batch_inputs["input_ids"].to(fresh_device),
                attention_mask=batch_inputs["attention_mask"].to(fresh_device),
            )

        pretrained_decoded = sentencepiece_tokenizer.batch_decode(
            pretrained_outputs, skip_special_tokens=True
        )
        fresh_decoded = sentencepiece_tokenizer.batch_decode(
            fresh_outputs, skip_special_tokens=True
        )

        for buggy_method, pred_a, pred_b in zip(batch_buggy, pretrained_decoded, fresh_decoded):
            predictions[buggy_method]["pretrained_finetuned_t5"] = pred_a
            predictions[buggy_method]["fresh_finetuned_t5"] = pred_b

    # Qwen prediciton generation pass separately
    # Note Qwen passed to device inside load_qwen_generator sub method
    for buggy_method in tqdm(buggy_methods, desc="Qwen generation"):
        predictions[buggy_method]["qwen_zero_shot"] = generate_code(
            buggy_method, llm_config, mode="zero_shot"
        )[0]
        predictions[buggy_method]["qwen_rag"] = generate_code(
            buggy_method, llm_config, mode="RAG"
        )[0]

    with open(predictions_path, "w", encoding="utf-8") as f:
        json.dump(predictions, f, indent=2, ensure_ascii=False)
    print(f"Saved predictions to {predictions_path}")
else:
    print(f"Predictions file found. Loading predictions from {predictions_path}...")
    with open(predictions_path, "r", encoding="utf-8") as f:
        predictions = json.load(f)


README.md: 0.00B [00:00, ?B/s]

medium/train-00000-of-00001.parquet:   0%|          | 0.00/11.9M [00:00<?, ?B/s]

medium/validation-00000-of-00001.parquet:   0%|          | 0.00/1.50M [00:00<?, ?B/s]

medium/test-00000-of-00001.parquet:   0%|          | 0.00/1.49M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/52364 [00:00<?, ? examples/s]

...

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

T5 generation:   0%|          | 0/63 [00:00<?, ?it/s]

Qwen generation:   0%|          | 0/500 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
/content/multi-paradigm-bug-fixing/src/llm_utils.py:164: RuntimeWarning: Skipping record -1: query buggy method could not be parsed as Java
  _warn_on_skipped_record(-1, "query buggy method could not be parsed as Java")


Saved predictions to /content/drive/MyDrive/multi-paradigm-bug-fixing/outputs/predictions/predictions.json


### 10ii. Evaluate on select Metrics and Save Predictions/Results:
- Exact Match
- CodeBLEU
- Create result table
- save predictions and results

In [16]:
import pandas as pd
from codebleu import calc_codebleu

model_names = [
    "pretrained_finetuned_t5",
    "fresh_finetuned_t5",
    "qwen_zero_shot",
    "qwen_rag",
]

row_records = []

for model_name in model_names:
    references = []
    predictions_for_model = []

    for buggy_code, fixed_code in test_pairs:
        pred = predictions[buggy_code][model_name]
        references.append(fixed_code)
        predictions_for_model.append(pred)

    exact_matches = [
        (pred.strip()) == (ref.strip())
        for pred, ref in zip(predictions_for_model, references)
    ]

    #using average exact match here, sum of matches per pred/fixed pair over all comparisons
    exact_match_score = sum(exact_matches) / len(exact_matches)

    #calculate codebleu from codebelu package
    codebleu_result = calc_codebleu(
        references=references,
        predictions=predictions_for_model,
        lang="java",
    )

    row_records.append({
        "Model": model_name,
        "ExactMatch": round(exact_match_score, 4),
        "CodeBLEU": round(codebleu_result["codebleu"], 4),
    })

results_df = pd.DataFrame(row_records).sort_values(
    by=["ExactMatch", "CodeBLEU"],
    ascending=False,
).reset_index(drop=True)

display(results_df)

#save results to RESULTS_DIR as CSV
results_csv_path = RESULTS_DIR / "evaluation_results.csv"

results_df.to_csv(results_csv_path, index=False)

print(f"Saved results table to {results_csv_path}")

AssertionError: Number of references and predictions should be the same